**Linking to kaggle for dataset**

In [ ]:
from google.colab import drive
from google.colab import files
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folders in your Drive
os.makedirs('/content/drive/MyDrive/fl-plant-disease/data', exist_ok=True)

# Upload your kaggle.json file
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Move the key to the correct hidden folder for Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API key configured successfully!")


Mounted at /content/drive
Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API key configured successfully!


In [ ]:
# Move to your Drive data folder
%cd /content/drive/MyDrive/fl-plant-disease/data/

# Download the dataset
!kaggle datasets download -d emmarex/plantdisease


/content/drive/MyDrive/fl-plant-disease/data
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:04<00:00, 168MB/s]



In [1]:
!ls /content/drive/MyDrive/fl-plant-disease/data/

ls: cannot access '/content/drive/MyDrive/fl-plant-disease/data/': No such file or directory


**At first mount the drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Secondly, run this to load the dataset in Colab local storage**

In [4]:
# 1. Copy the zip file from Drive to local Colab storage
!cp /content/drive/MyDrive/fl-plant-disease/data/plantdisease.zip /content/plantdisease.zip

# 2. Unzip quietly (-q) to the /content/dataset folder
!unzip -q /content/plantdisease.zip -d /content/dataset/

# 3. Remove the local zip file to save space
!rm /content/plantdisease.zip

# 4. Check if it worked!
!ls /content/dataset/


plantvillage  PlantVillage


In [ ]:
# Check the contents of the PlantVillage folder
!ls /content/dataset/PlantVillage


Pepper__bell___Bacterial_spot  Tomato_Late_blight
Pepper__bell___healthy	       Tomato_Leaf_Mold
Potato___Early_blight	       Tomato_Septoria_leaf_spot
Potato___healthy	       Tomato_Spider_mites_Two_spotted_spider_mite
Potato___Late_blight	       Tomato__Target_Spot
Tomato_Bacterial_spot	       Tomato__Tomato_mosaic_virus
Tomato_Early_blight	       Tomato__Tomato_YellowLeaf__Curl_Virus
Tomato_healthy


In [5]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import os

# 1. Define the path to your images
# (Adjust this if Step 1 shows a different path, e.g., /content/dataset/plantvillage)
data_dir = '/content/dataset/PlantVillage'

# 2. Define Image Transformations
# Neural networks need images to be the same size and converted to Tensors.
# We use standard ImageNet normalization values.
transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Standard size for MobileNet/EfficientNet
    transforms.ToTensor(),               # Convert image to PyTorch Tensor
    transforms.Normalize(                # Normalize pixel values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Load the Dataset
# ImageFolder automatically assigns labels based on folder names
full_dataset = torchvision.datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images found: {len(full_dataset)}")
print(f"Total classes found: {len(full_dataset.classes)}")

# 4. Split into Train (80%) and Validation (20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

# 5. Create DataLoaders
# DataLoaders handle batching, shuffling, and loading data in parallel
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("DataLoaders are ready!")


Total images found: 20638
Total classes found: 15
Training images: 16510
Validation images: 4128
DataLoaders are ready!


In [ ]:
# Grab one batch of training data
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}") # Expected: [32, 3, 224, 224] (Batch Size, Channels, Height, Width)
print(f"Label batch shape: {labels.shape}") # Expected: [32]


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


**Now that the data pipeline is ready, we need to train a standard, non-federated model to prove the dataset works. We will use MobileNetV2 because it is lightweight and perfect for Federated Learning later.**

In [9]:
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# 1. Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load pre-trained MobileNetV2
# Using pre-trained weights speeds up training significantly (Transfer Learning)
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)

# 3. Modify the final classification layer for our 15 classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 15)

model = model.to(device)

# 4. Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model, Loss, and Optimizer initialized successfully!")

Using device: cuda
Model, Loss, and Optimizer initialized successfully!


**Training MobileNetV2 model**

In [10]:
import time
import torch

# Let's train for 5 epochs to establish a baseline
num_epochs = 5

for epoch in range(num_epochs):
    start_time = time.time()
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    # --- TRAINING PHASE ---
    model.train()  # Set model to training mode
    running_loss = 0.0
    running_corrects = 0

    # Iterate over data
    for inputs, labels in train_loader:
        # Move inputs and labels to the GPU
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    # --- VALIDATION PHASE ---
    model.eval()   # Set model to evaluate mode (disables dropout, etc.)
    val_loss = 0.0
    val_corrects = 0

    # Disable gradient calculation for validation (saves memory and computes faster)
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = val_corrects.double() / len(val_dataset)

    end_time = time.time()
    epoch_time = end_time - start_time

    print(f'Val Loss:   {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')
    print(f'Time taken: {epoch_time:.0f} seconds\n')

print('Baseline training complete!')


Epoch 1/5
----------
Train Loss: 0.2774 Acc: 0.9149
Val Loss:   0.0874 Acc: 0.9704
Time taken: 85 seconds

Epoch 2/5
----------
Train Loss: 0.0953 Acc: 0.9695
Val Loss:   0.0657 Acc: 0.9797
Time taken: 89 seconds

Epoch 3/5
----------
Train Loss: 0.0835 Acc: 0.9720
Val Loss:   0.2395 Acc: 0.9305
Time taken: 89 seconds

Epoch 4/5
----------
Train Loss: 0.0557 Acc: 0.9810
Val Loss:   0.0542 Acc: 0.9818
Time taken: 88 seconds

Epoch 5/5
----------
Train Loss: 0.0675 Acc: 0.9786
Val Loss:   0.0976 Acc: 0.9729
Time taken: 91 seconds

Baseline training complete!


# **Federated Learning with Flower (flwr)**

In [6]:
!pip install flwr["simulation"]


In [7]:
# Number of simulated FL clients
NUM_CLIENTS = 5

# Calculate the size of each partition
partition_size = len(train_dataset) // NUM_CLIENTS
lengths = [partition_size] * NUM_CLIENTS
# Add any remainder to the last client
lengths[-1] += len(train_dataset) % NUM_CLIENTS

# Split the dataset
client_datasets = random_split(train_dataset, lengths)

for i, ds in enumerate(client_datasets):
    print(f"Client {i+1} gets {len(ds)} training images.")


Client 1 gets 3302 training images.
Client 2 gets 3302 training images.
Client 3 gets 3302 training images.
Client 4 gets 3302 training images.
Client 5 gets 3302 training images.
